# Week 42, Day 3: The BCM rule: a sliding threshold for selectivity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w042/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A fixed threshold fails

Write bcm_fixed(theta, steps=4000): present x1 or x2 at random on each step, compute y, update the weights by eta x y (y - theta) and keep them non-negative; stop early if the weight norm exceeds 100. Run it with theta 0.4 and 1.0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1982)
x1 = np.array([1.0, 0.8, 0.2, 0.1]); x2 = np.array([0.1, 0.2, 0.8, 1.0])
eta = 0.005

def bcm_fixed(theta, steps=4000):
    w = np.full(4, 0.3)
    for t in range(steps):
        x = x1 if r.random() < 0.5 else x2
        y = max(0.0, w @ x)
        ...
        if np.linalg.norm(w) > 100:
            break
    return w

w_low, w_high = bcm_fixed(0.4), bcm_fixed(1.0)
print(f"theta 0.4: |w| = {np.linalg.norm(w_low):.3g};  theta 1.0: |w| = {np.linalg.norm(w_high):.3g}  (initial response about {0.3 * x1.sum():.2f})")

In [ ]:
import numpy as np
def test_low():
    "with a fixed threshold of 0.4, the weights blow up"
    assert np.linalg.norm(w_low) > 50

def test_high():
    "with a fixed threshold of 1.0, the weights collapse to near zero"
    assert np.linalg.norm(w_high) < 0.05

test_low()
test_high()
print("All checks passed")

### Exercise: Selectivity from a sliding threshold

Write bcm_sliding(steps=20000): as before, but update theta after each step by (y² / y0 - theta) / tau_theta, starting from theta = 0.5. Return the weights, the final theta and the history of the two responses (every 100 steps).

In [ ]:
def bcm_sliding(steps=20000, tau_theta=50.0, y0=1.0):
    w = np.full(4, 0.3); theta = 0.5; hist = []
    for t in range(steps):
        x = x1 if r.random() < 0.5 else x2
        y = max(0.0, w @ x)
        ...
        if t % 100 == 0:
            hist.append([max(0, w @ x1), max(0, w @ x2), theta])
    return w, theta, np.array(hist)

w_s, theta_s, hist = bcm_sliding()
print(f"weights {np.round(w_s, 2)}; response to x1 {max(0, w_s @ x1):.2f}, to x2 {max(0, w_s @ x2):.2f}; theta {theta_s:.2f}")
plt.plot(hist[:, 0], label="response to x1"); plt.plot(hist[:, 1], label="response to x2"); plt.plot(hist[:, 2], "k:", label="theta")
plt.xlabel("hundreds of presentations"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_selective():
    "the neuron responds at least five times more to one pattern than to the other"
    a, b = max(0, w_s @ x1), max(0, w_s @ x2)
    assert max(a, b) > 5 * min(a, b)

def test_stable():
    "the threshold settles near the mean of y squared"
    m = 0.5 * (max(0, w_s @ x1) ** 2 + max(0, w_s @ x2) ** 2)
    assert abs(theta_s - m) < 0.25 * m

test_selective()
test_stable()
print("All checks passed")

### Exercise: History changes the sign of plasticity

Let theta settle under 500 steps of constant activity y = 1.5 (an active history) or y = 0.3 (a quiet history), using the theta update alone. Then compute the change in a synapse with x = 1 for a test response y = 0.8 under each threshold.

In [ ]:
def settle_theta(y_hist, steps=500, tau_theta=50.0, theta=0.5):
    for _ in range(steps):
        theta += (y_hist ** 2 - theta) / tau_theta
    return theta

theta_active, theta_quiet = settle_theta(1.5), settle_theta(0.3)
y_test = 0.8
dw_active = ...
dw_quiet = ...
print(f"theta after an active history {theta_active:.2f}, after a quiet one {theta_quiet:.2f}")
print(f"change for the same test response: {dw_active:+.4f} (active history), {dw_quiet:+.4f} (quiet history)")

In [ ]:
import numpy as np
def test_sign():
    "the same test response potentiates after quiet history and depresses after active history"
    assert dw_quiet > 0 > dw_active

test_sign()
print("All checks passed")